# Text to SQL Evaluation

Evaluating LLM generated SQL: Exact Match, Component Matching, Execution Accuracy (SQLite), SQL Complexity Analysis, Valid Efficiency Score (VES), Error Categorization, Schema Linking, and End to End generation.

In [ ]:
GROQ_MODEL_FAST  = "llama-3.1-8b-instant"
GROQ_MODEL_SMART = "llama-3.3-70b-versatile"
OPENAI_BASE_URL = "https://api.groq.com/openai/v1"

import os
import json, re, time
import numpy as np
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def groq_chat(prompt, system="You are a helpful assistant.",
              model=GROQ_MODEL_SMART, temperature=0.0, max_tokens=600):
    resp = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system",  "content": system},
            {"role": "user",    "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return resp.choices[0].message.content.strip()

def parse_json(raw):
    raw = re.sub(r'```json|```', '', raw).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(m.group()) if m else {}

def show(title, d):
    print(f"\n{'='*60}\n  📊 {title}\n{'='*60}")
    for k, v in d.items():
        print(f"  {k:<30}: {v:.4f}" if isinstance(v, float) else f"  {k:<30}: {v}")

print('🤖 Groq test:', groq_chat('Reply with only: Groq is ready!', max_tokens=20))

✅ All packages installed!
🤖 Groq test: Groq is ready!


## 1. Exact Match and Component Matching
### Tool: custom SQL component parser

In [2]:
def normalize_sql(sql):
    """Normalize SQL for comparison: lowercase, collapse whitespace, strip semicolons."""
    sql = re.sub(r'--.*', '', sql)  # remove comments
    sql = re.sub(r'\s+', ' ', sql.lower().strip().rstrip(';'))
    # Normalize quotes
    sql = sql.replace('"', "'")
    return sql

def sql_components(sql):
    """Extract SQL clause components for partial matching."""
    s = sql.lower()
    sm  = re.search(r'select\s+(.*?)\s+from', s, re.DOTALL)
    fm  = re.search(r'from\s+([\w,\s]+?)(?:\s+where|\s+join|\s+group|\s+order|\s+having|\s+limit|$)', s)
    wm  = re.search(r'where\s+(.*?)(?:\s+group|\s+order|\s+having|\s+limit|$)', s, re.DOTALL)
    gm  = re.search(r'group\s+by\s+(.*?)(?:\s+having|\s+order|\s+limit|$)', s, re.DOTALL)
    om  = re.search(r'order\s+by\s+(.*?)(?:\s+limit|$)', s, re.DOTALL)
    hm  = re.search(r'having\s+(.*?)(?:\s+order|\s+limit|$)', s, re.DOTALL)
    jns = re.findall(r'\w*\s*join\s+\w+', s)
    return {
        'select':   sm.group(1).strip() if sm else '',
        'from':     fm.group(1).strip() if fm else '',
        'where':    wm.group(1).strip() if wm else '',
        'group_by': gm.group(1).strip() if gm else '',
        'order_by': om.group(1).strip() if om else '',
        'having':   hm.group(1).strip() if hm else '',
        'joins':    jns,
    }

sql_pairs = [
    {'q': 'Find all users older than 25',
     'pred': 'SELECT * FROM users WHERE age > 25;',
     'gold': 'SELECT * FROM users WHERE age > 25'},
    {'q': 'Count orders per customer',
     'pred': 'SELECT customer_id, COUNT(*) as total FROM orders GROUP BY customer_id',
     'gold': 'SELECT customer_id, COUNT(id) as order_count FROM orders GROUP BY customer_id'},
    {'q': 'Get user names and their orders',
     'pred': 'SELECT u.name, o.id FROM users u JOIN orders o ON u.id = o.user_id',
     'gold': 'SELECT u.name, o.id FROM users u INNER JOIN orders o ON u.id = o.user_id'},
    {'q': 'Top 5 products by revenue',
     'pred': 'SELECT product, SUM(amount) as rev FROM orders GROUP BY product ORDER BY rev DESC LIMIT 5',
     'gold': 'SELECT product, SUM(amount) as revenue FROM orders GROUP BY product ORDER BY revenue DESC LIMIT 5'},
    {'q': 'Users with more than 3 orders',
     'pred': 'SELECT user_id, COUNT(*) as cnt FROM orders GROUP BY user_id HAVING cnt > 3',
     'gold': 'SELECT user_id, COUNT(*) as cnt FROM orders GROUP BY user_id HAVING COUNT(*) > 3'},
]

print('\n🗄️  Exact Match & Component Matching')
print('='*70)
exact_hits, comp_scores = 0, []
clause_names = ['select', 'from', 'where', 'group_by', 'order_by']

for p in sql_pairs:
    pn, gn  = normalize_sql(p['pred']), normalize_sql(p['gold'])
    exact   = pn == gn
    if exact: exact_hits += 1
    pc, gc  = sql_components(pn), sql_components(gn)
    cm      = {k: pc[k]==gc[k] for k in clause_names}
    cs      = sum(cm.values()) / len(clause_names)
    comp_scores.append(cs)
    print(f'\n  Q: {p["q"]}')
    print(f'  Exact Match: {"✅" if exact else "❌"}  Component Score: {cs:.1%}')
    for clause in clause_names:
        if pc[clause] or gc[clause]:
            icon = "✅" if cm[clause] else "❌"
            print(f'    {icon} {clause.upper():<10}: pred="{pc[clause][:30]}"  gold="{gc[clause][:30]}"')

print(f'\n  Overall Exact Match  : {exact_hits/len(sql_pairs):.1%}')
print(f'  Avg Component Score  : {np.mean(comp_scores):.1%}')


🗄️  Exact Match & Component Matching

  Q: Find all users older than 25
  Exact Match: ✅  Component Score: 100.0%
    ✅ SELECT    : pred="*"  gold="*"
    ✅ FROM      : pred="users"  gold="users"
    ✅ WHERE     : pred="age > 25"  gold="age > 25"

  Q: Count orders per customer
  Exact Match: ❌  Component Score: 80.0%
    ❌ SELECT    : pred="customer_id, count(*) as total"  gold="customer_id, count(id) as orde"
    ✅ FROM      : pred="orders"  gold="orders"
    ✅ GROUP_BY  : pred="customer_id"  gold="customer_id"

  Q: Get user names and their orders
  Exact Match: ❌  Component Score: 80.0%
    ✅ SELECT    : pred="u.name, o.id"  gold="u.name, o.id"
    ❌ FROM      : pred="users u"  gold="users u inner"

  Q: Top 5 products by revenue
  Exact Match: ❌  Component Score: 60.0%
    ❌ SELECT    : pred="product, sum(amount) as rev"  gold="product, sum(amount) as revenu"
    ✅ FROM      : pred="orders"  gold="orders"
    ✅ GROUP_BY  : pred="product"  gold="product"
    ❌ ORDER_BY  : pred="re

## 2. Execution Accuracy (in-memory SQLite)
### Tool: SQLite — compare actual query results

In [3]:
import sqlite3

def make_db():
    conn = sqlite3.connect(':memory:')
    conn.execute('CREATE TABLE users (id INT, name TEXT, age INT, city TEXT)')
    conn.execute('CREATE TABLE orders (id INT, user_id INT, product TEXT, amount FLOAT)')
    conn.execute('CREATE TABLE products (id INT, name TEXT, category TEXT, price FLOAT)')
    conn.executemany('INSERT INTO users VALUES (?,?,?,?)', [
        (1,'Alice',30,'New York'),(2,'Bob',22,'Chicago'),
        (3,'Carol',35,'New York'),(4,'Dave',28,'LA'),
        (5,'Eve',40,'Chicago'),(6,'Frank',25,'New York')])
    conn.executemany('INSERT INTO orders VALUES (?,?,?,?)', [
        (1,1,'Laptop',999.99),(2,1,'Mouse',29.99),
        (3,2,'Keyboard',79.99),(4,3,'Monitor',399.99),
        (5,1,'Headphones',149.99),(6,4,'Mouse',29.99),
        (7,5,'Laptop',999.99),(8,3,'Keyboard',79.99)])
    conn.executemany('INSERT INTO products VALUES (?,?,?,?)', [
        (1,'Laptop','Electronics',999.99),(2,'Mouse','Accessories',29.99),
        (3,'Keyboard','Accessories',79.99),(4,'Monitor','Electronics',399.99),
        (5,'Headphones','Audio',149.99)])
    conn.commit()
    return conn

def run_sql(conn, sql):
    try:
        rows = conn.execute(sql).fetchall()
        return sorted([tuple(r) for r in rows]), None
    except Exception as e:
        return None, str(e)

conn = make_db()

exec_tests = [
    {'q': 'Users older than 25',     'pred': 'SELECT id, name FROM users WHERE age > 25',              'gold': 'SELECT id, name FROM users WHERE age > 25'},
    {'q': 'Users in New York',       'pred': "SELECT name FROM users WHERE city = 'New York'",         'gold': "SELECT name FROM users WHERE city = 'New York'"},
    {'q': 'Total spend per user',    'pred': 'SELECT user_id, SUM(amount) FROM orders',               'gold': 'SELECT user_id, SUM(amount) FROM orders GROUP BY user_id'},
    {'q': 'Products for user 1',     'pred': 'SELECT product FROM orders WHERE user_id = 1',           'gold': 'SELECT product FROM orders WHERE user_id = 1'},
    {'q': 'Count orders per city',   'pred': 'SELECT u.city, COUNT(o.id) FROM users u JOIN orders o ON u.id = o.user_id GROUP BY u.city',
                                     'gold': 'SELECT u.city, COUNT(o.id) FROM users u JOIN orders o ON u.id = o.user_id GROUP BY u.city'},
    {'q': 'Avg order amount',        'pred': 'SELECT AVG(amount) FROM orders',                         'gold': 'SELECT AVG(amount) FROM orders'},
    {'q': 'Max order amount per user','pred': 'SELECT user_id, MAX(amount) FROM orders GROUP BY user_id', 'gold': 'SELECT user_id, MAX(amount) FROM orders GROUP BY user_id'},
]

print('\n⚡ Execution Accuracy (SQLite)')
print('='*60)
correct = 0
for t in exec_tests:
    pr, pe = run_sql(conn, t['pred'])
    gr, _  = run_sql(conn, t['gold'])
    ok = (pr == gr) and pe is None
    if ok: correct += 1
    print(f'  {"✅" if ok else "❌"}  {t["q"]}')
    if not ok:
        print(f'     Gold: {gr}')
        print(f'     Got : {pr}  err={pe}')
print(f'\n  Execution Accuracy: {correct}/{len(exec_tests)} = {correct/len(exec_tests):.1%}')


⚡ Execution Accuracy (SQLite)
  ✅  Users older than 25
  ✅  Users in New York
  ❌  Total spend per user
     Gold: [(1, 1179.97), (2, 79.99), (3, 479.98), (4, 29.99), (5, 999.99)]
     Got : [(1, 2769.92)]  err=None
  ✅  Products for user 1
  ✅  Count orders per city
  ✅  Avg order amount
  ✅  Max order amount per user

  Execution Accuracy: 6/7 = 85.7%


## 3. SQL Complexity Analysis
### Tool: Custom SQL complexity scorer

In [4]:
def sql_complexity(sql):
    """Analyze the complexity of a SQL query."""
    s = sql.lower()
    features = {
        'has_join':        bool(re.search(r'\bjoin\b', s)),
        'has_subquery':    bool(re.search(r'select.*select', s)),
        'has_aggregation': bool(re.search(r'\b(count|sum|avg|min|max)\s*\(', s)),
        'has_group_by':    bool(re.search(r'group\s+by', s)),
        'has_having':      bool(re.search(r'\bhaving\b', s)),
        'has_order_by':    bool(re.search(r'order\s+by', s)),
        'has_limit':       bool(re.search(r'\blimit\b', s)),
        'has_distinct':    bool(re.search(r'\bdistinct\b', s)),
        'has_union':       bool(re.search(r'\bunion\b', s)),
        'has_case_when':   bool(re.search(r'\bcase\b.*\bwhen\b', s)),
        'has_like':        bool(re.search(r'\blike\b', s)),
        'has_in':          bool(re.search(r'\bin\s*\(', s)),
        'has_between':     bool(re.search(r'\bbetween\b', s)),
        'num_tables':      len(set(re.findall(r'(?:from|join)\s+(\w+)', s))),
        'num_conditions':  len(re.findall(r'\b(and|or)\b', s)) + (1 if re.search(r'\bwhere\b', s) else 0),
    }
    
    # Complexity score (0-10 scale)
    score = sum([
        features['has_join'] * 2,
        features['has_subquery'] * 3,
        features['has_aggregation'] * 1,
        features['has_group_by'] * 1,
        features['has_having'] * 1.5,
        features['has_order_by'] * 0.5,
        features['has_distinct'] * 0.5,
        features['has_union'] * 2,
        features['has_case_when'] * 2,
        min(features['num_tables'], 4) * 0.5,
        min(features['num_conditions'], 4) * 0.3,
    ])
    
    level = 'simple' if score < 2 else 'moderate' if score < 4 else 'complex' if score < 7 else 'very complex'
    return {**features, 'complexity_score': score, 'complexity_level': level}

test_queries = [
    "SELECT * FROM users WHERE age > 25",
    "SELECT u.name, COUNT(o.id) FROM users u JOIN orders o ON u.id = o.user_id GROUP BY u.name",
    "SELECT city, AVG(amount) FROM users u JOIN orders o ON u.id = o.user_id GROUP BY city HAVING AVG(amount) > 100 ORDER BY AVG(amount) DESC",
    "SELECT * FROM users WHERE id IN (SELECT user_id FROM orders WHERE amount > (SELECT AVG(amount) FROM orders))",
    "SELECT CASE WHEN age < 25 THEN 'young' WHEN age < 35 THEN 'mid' ELSE 'senior' END as age_group, COUNT(*) FROM users GROUP BY age_group",
]

print('\n🔬 SQL Complexity Analysis')
print('='*70)
for sql in test_queries:
    cx = sql_complexity(sql)
    flags = [k.replace('has_','') for k, v in cx.items() if isinstance(v, bool) and v]
    print(f'\n  SQL: {sql[:70]}...' if len(sql)>70 else f'\n  SQL: {sql}')
    print(f'  Complexity: {cx["complexity_score"]:.1f}/10 ({cx["complexity_level"]})')
    print(f'  Features: {", ".join(flags) if flags else "basic"} | Tables: {cx["num_tables"]} | Conditions: {cx["num_conditions"]}')


🔬 SQL Complexity Analysis

  SQL: SELECT * FROM users WHERE age > 25
  Complexity: 0.8/10 (simple)
  Features: basic | Tables: 1 | Conditions: 1

  SQL: SELECT u.name, COUNT(o.id) FROM users u JOIN orders o ON u.id = o.user...
  Complexity: 5.0/10 (complex)
  Features: join, aggregation, group_by | Tables: 2 | Conditions: 0

  SQL: SELECT city, AVG(amount) FROM users u JOIN orders o ON u.id = o.user_i...
  Complexity: 7.0/10 (very complex)
  Features: join, aggregation, group_by, having, order_by | Tables: 2 | Conditions: 0

  SQL: SELECT * FROM users WHERE id IN (SELECT user_id FROM orders WHERE amou...
  Complexity: 5.3/10 (complex)
  Features: subquery, aggregation, in | Tables: 2 | Conditions: 1

  SQL: SELECT CASE WHEN age < 25 THEN 'young' WHEN age < 35 THEN 'mid' ELSE '...
  Complexity: 4.5/10 (complex)
  Features: aggregation, group_by, case_when | Tables: 1 | Conditions: 0


## 4. Valid Efficiency Score (VES)
### Tool: SQLite + timing — measures both correctness AND execution efficiency

In [5]:
def valid_efficiency_score(conn, pred_sql, gold_sql, n_runs=5):
    """Compute VES: only valid (correct) queries get efficiency credit."""
    pr, pe = run_sql(conn, pred_sql)
    gr, _  = run_sql(conn, gold_sql)
    
    is_valid = (pr == gr) and pe is None
    if not is_valid:
        return {'valid': False, 'ves': 0.0, 'pred_time': None, 'gold_time': None}
    
    # Time the execution
    def time_query(sql):
        times = []
        for _ in range(n_runs):
            start = time.perf_counter()
            conn.execute(sql).fetchall()
            times.append(time.perf_counter() - start)
        return np.median(times)
    
    pred_time = time_query(pred_sql)
    gold_time = time_query(gold_sql)
    
    # VES = sqrt(gold_time / pred_time), capped at 1.0 (no bonus for being faster)
    ves = min(1.0, np.sqrt(gold_time / pred_time)) if pred_time > 0 else 1.0
    
    return {'valid': True, 'ves': ves, 'pred_time': pred_time, 'gold_time': gold_time}

ves_tests = [
    {'q': 'Users older than 25',
     'pred': 'SELECT id, name FROM users WHERE age > 25',
     'gold': 'SELECT id, name FROM users WHERE age > 25'},
    {'q': 'Products for user 1',
     'pred': 'SELECT product FROM orders WHERE user_id = 1',
     'gold': 'SELECT product FROM orders WHERE user_id = 1'},
    {'q': 'Total spend (wrong query)',
     'pred': 'SELECT SUM(amount) FROM orders WHERE user_id = 999',
     'gold': 'SELECT SUM(amount) FROM orders'},
]

print('\n⏱️  Valid Efficiency Score (VES)')
print('='*65)
ves_scores = []
for t in ves_tests:
    result = valid_efficiency_score(conn, t['pred'], t['gold'])
    ves_scores.append(result['ves'])
    if result['valid']:
        print(f'  ✅ {t["q"]}')
        print(f'     VES={result["ves"]:.4f}  pred={result["pred_time"]*1000:.3f}ms  gold={result["gold_time"]*1000:.3f}ms')
    else:
        print(f'  ❌ {t["q"]}  (invalid result → VES=0)')
print(f'\n  Average VES: {np.mean(ves_scores):.4f}')


⏱️  Valid Efficiency Score (VES)
  ✅ Users older than 25
     VES=1.0000  pred=0.002ms  gold=0.002ms
  ✅ Products for user 1
     VES=0.9917  pred=0.002ms  gold=0.002ms
  ❌ Total spend (wrong query)  (invalid result → VES=0)

  Average VES: 0.6639


## 5. SQL Error Categorization
### Tool: Custom error classifier for debugging common LLM SQL mistakes

In [6]:
def categorize_sql_error(pred_sql, gold_sql, conn):
    """Classify the type of error in a predicted SQL query."""
    errors = []
    
    # Check syntax
    try:
        conn.execute(f"EXPLAIN QUERY PLAN {pred_sql}")
    except Exception as e:
        errors.append(("SYNTAX_ERROR", str(e)))
        return errors
    
    pn, gn = normalize_sql(pred_sql), normalize_sql(gold_sql)
    pc, gc = sql_components(pn), sql_components(gn)
    
    # Check each clause
    if pc['select'] != gc['select']:
        errors.append(("WRONG_SELECT", f"pred='{pc['select'][:40]}' gold='{gc['select'][:40]}'"))
    if pc['from'] != gc['from']:
        errors.append(("WRONG_TABLE", f"pred='{pc['from'][:30]}' gold='{gc['from'][:30]}'"))
    if pc['where'] != gc['where']:
        errors.append(("WRONG_WHERE", f"pred='{pc['where'][:40]}' gold='{gc['where'][:40]}'"))
    if pc['group_by'] != gc['group_by']:
        errors.append(("WRONG_GROUPBY", f"pred='{pc['group_by'][:30]}' gold='{gc['group_by'][:30]}'"))
    if pc['joins'] != gc['joins']:
        errors.append(("WRONG_JOIN", f"pred={pc['joins']} gold={gc['joins']}"))
    
    # Check execution result
    pr, pe = run_sql(conn, pred_sql)
    gr, _  = run_sql(conn, gold_sql)
    if pe:
        errors.append(("RUNTIME_ERROR", pe[:60]))
    elif pr != gr:
        errors.append(("WRONG_RESULT", f"got {len(pr or [])} rows vs expected {len(gr or [])} rows"))
    
    if not errors:
        errors.append(("NONE", "Query is correct"))
    
    return errors

error_test_queries = [
    {'q': 'Total spend per user', 
     'pred': 'SELECT user_id, SUM(amount) FROM orders',
     'gold': 'SELECT user_id, SUM(amount) FROM orders GROUP BY user_id'},
    {'q': 'Users in NYC',
     'pred': "SELECT * FROM user WHERE city = 'New York'",  # wrong table name
     'gold': "SELECT * FROM users WHERE city = 'New York'"},
    {'q': 'Avg order over 100',
     'pred': 'SELECT user_id, AVG(amount) FROM orders GROUP BY user_id WHERE AVG(amount) > 100',  # syntax error
     'gold': 'SELECT user_id, AVG(amount) FROM orders GROUP BY user_id HAVING AVG(amount) > 100'},
    {'q': 'User count by city',
     'pred': 'SELECT city, COUNT(*) as cnt FROM users GROUP BY city ORDER BY cnt DESC',
     'gold': 'SELECT city, COUNT(*) as cnt FROM users GROUP BY city ORDER BY cnt DESC'},
]

print('\n🔍 SQL Error Categorization')
print('='*65)
error_counts = {}
for t in error_test_queries:
    errors = categorize_sql_error(t['pred'], t['gold'], conn)
    print(f'\n  Q: {t["q"]}')
    print(f'  Pred: {t["pred"][:60]}')
    for etype, detail in errors:
        error_counts[etype] = error_counts.get(etype, 0) + 1
        icon = "✅" if etype == "NONE" else "🔴"
        print(f'  {icon} {etype}: {detail[:60]}')

print(f'\n  Error Distribution:')
for etype, count in sorted(error_counts.items(), key=lambda x: -x[1]):
    bar = "█" * count
    print(f'    {etype:<20} {bar} ({count})')


🔍 SQL Error Categorization

  Q: Total spend per user
  Pred: SELECT user_id, SUM(amount) FROM orders
  🔴 WRONG_GROUPBY: pred='' gold='user_id'
  🔴 WRONG_RESULT: got 1 rows vs expected 5 rows

  Q: Users in NYC
  Pred: SELECT * FROM user WHERE city = 'New York'
  🔴 SYNTAX_ERROR: no such table: user

  Q: Avg order over 100
  Pred: SELECT user_id, AVG(amount) FROM orders GROUP BY user_id WHE
  🔴 SYNTAX_ERROR: near "WHERE": syntax error

  Q: User count by city
  Pred: SELECT city, COUNT(*) as cnt FROM users GROUP BY city ORDER 
  ✅ NONE: Query is correct

  Error Distribution:
    SYNTAX_ERROR         ██ (2)
    WRONG_GROUPBY        █ (1)
    WRONG_RESULT         █ (1)
    NONE                 █ (1)


## 6. Schema Linking Evaluation
### Tool: Checks if the LLM correctly identifies relevant tables/columns

In [7]:
SCHEMA_LINK_SYS = '''Given the database schema and a question, identify which tables and columns are needed.
Schema: users(id, name, age, city), orders(id, user_id, product, amount), products(id, name, category, price)
Return JSON only: {"tables": ["table1", ...], "columns": ["table.column", ...]}'''

schema_link_tests = [
    {
        "question": "What are the names of users who ordered laptops?",
        "gold_tables": {"users", "orders"},
        "gold_columns": {"users.name", "orders.product", "users.id", "orders.user_id"},
    },
    {
        "question": "What is the average order amount by city?",
        "gold_tables": {"users", "orders"},
        "gold_columns": {"users.city", "orders.amount", "users.id", "orders.user_id"},
    },
    {
        "question": "List all products in the Electronics category.",
        "gold_tables": {"products"},
        "gold_columns": {"products.name", "products.category"},
    },
]

print('\n🔗 Schema Linking Evaluation')
print('='*65)
sl_scores = []
for test in schema_link_tests:
    result = parse_json(groq_chat(test["question"], system=SCHEMA_LINK_SYS))
    pred_tables = set(t.lower() for t in result.get("tables", []))
    pred_cols = set(c.lower() for c in result.get("columns", []))
    gold_tables = set(t.lower() for t in test["gold_tables"])
    gold_cols = set(c.lower() for c in test["gold_columns"])
    
    table_recall = len(pred_tables & gold_tables) / len(gold_tables) if gold_tables else 1.0
    table_precision = len(pred_tables & gold_tables) / len(pred_tables) if pred_tables else 0.0
    col_recall = len(pred_cols & gold_cols) / len(gold_cols) if gold_cols else 1.0
    
    sl_scores.append(table_recall)
    print(f'\n  Q: {test["question"]}')
    print(f'  Tables — Gold: {gold_tables}  Pred: {pred_tables}  Recall={table_recall:.1%}  Precision={table_precision:.1%}')
    print(f'  Columns — Gold: {gold_cols}')
    print(f'             Pred: {pred_cols}  Recall={col_recall:.1%}')

print(f'\n  Avg Table Recall: {np.mean(sl_scores):.1%}')


🔗 Schema Linking Evaluation

  Q: What are the names of users who ordered laptops?
  Tables — Gold: {'orders', 'users'}  Pred: {'products', 'orders', 'users'}  Recall=100.0%  Precision=66.7%
  Columns — Gold: {'users.id', 'orders.user_id', 'orders.product', 'users.name'}
             Pred: {'orders.user_id', 'products.name', 'orders.product', 'users.name'}  Recall=75.0%

  Q: What is the average order amount by city?
  Tables — Gold: {'orders', 'users'}  Pred: {'orders', 'users'}  Recall=100.0%  Precision=100.0%
  Columns — Gold: {'users.id', 'orders.user_id', 'users.city', 'orders.amount'}
             Pred: {'users.city', 'orders.amount'}  Recall=50.0%

  Q: List all products in the Electronics category.
  Tables — Gold: {'products'}  Pred: {'products'}  Recall=100.0%  Precision=100.0%
  Columns — Gold: {'products.name', 'products.category'}
             Pred: {'products.id', 'products.name', 'products.price', 'products.category'}  Recall=100.0%

  Avg Table Recall: 100.0%


## 7. Groq as Text-to-SQL Generator (End-to-End)
### Tool: Groq + SQLite execution

In [8]:
SCHEMA = 'Tables: users(id,name,age,city), orders(id,user_id,product,amount), products(id,name,category,price)'
SQL_SYS = f'You are a SQL expert for SQLite. Schema: {SCHEMA}. Return SQL only — no explanation.'

nl_questions = [
    ("List all users from New York",                      "SELECT * FROM users WHERE city = 'New York'"),
    ("How many orders does each user have?",               "SELECT user_id, COUNT(*) FROM orders GROUP BY user_id"),
    ("What is the most expensive product ordered?",       "SELECT product, MAX(amount) FROM orders"),
    ("Find users who have placed more than 1 order",      "SELECT user_id, COUNT(*) as cnt FROM orders GROUP BY user_id HAVING cnt > 1"),
    ("What is the average age of users in each city?",    "SELECT city, AVG(age) FROM users GROUP BY city"),
    ("List products ordered by Alice",                     "SELECT o.product FROM orders o JOIN users u ON o.user_id = u.id WHERE u.name = 'Alice'"),
    ("Which city has the highest total order amount?",    "SELECT u.city, SUM(o.amount) as total FROM users u JOIN orders o ON u.id = o.user_id GROUP BY u.city ORDER BY total DESC LIMIT 1"),
]

print('\n🤖 Groq Text-to-SQL + Execution Evaluation')
print('='*70)
correct = 0
complexities = []
for question, gold_sql in nl_questions:
    # Clean the predicted SQL
    pred_sql = groq_chat(question, system=SQL_SYS, model=GROQ_MODEL_SMART).strip()
    pred_sql = re.sub(r'```sql|```', '', pred_sql).strip().rstrip(';')
    
    pr, pe   = run_sql(conn, pred_sql)
    gr, _    = run_sql(conn, gold_sql)
    ok = (pr == gr) and pe is None
    if ok: correct += 1
    
    cx = sql_complexity(pred_sql)
    complexities.append(cx['complexity_level'])
    
    print(f'  {"✅" if ok else "❌"}  {question}')
    print(f'     SQL: {pred_sql[:70]}')
    print(f'     Complexity: {cx["complexity_level"]} ({cx["complexity_score"]:.1f})')
    if not ok:
        print(f'     Gold result: {str(gr)[:60]}')
        print(f'     Got result : {str(pr)[:60]}  err={pe}')

print(f'\n  Groq SQL Execution Accuracy: {correct}/{len(nl_questions)} = {correct/len(nl_questions):.1%}')
print(f'  Complexity distribution: {dict((k, complexities.count(k)) for k in set(complexities))}')


🤖 Groq Text-to-SQL + Execution Evaluation
  ✅  List all users from New York
     SQL: SELECT * 
FROM users 
WHERE city = 'New York'
     Complexity: simple (0.8)
  ❌  How many orders does each user have?
     SQL: SELECT u.id, u.name, COUNT(o.id) AS order_count
FROM users u
LEFT JOIN
     Complexity: complex (5.0)
     Gold result: [(1, 3), (2, 1), (3, 2), (4, 1), (5, 1)]
     Got result : [(1, 'Alice', 3), (2, 'Bob', 1), (3, 'Carol', 2), (4, 'Dave'  err=None
  ❌  What is the most expensive product ordered?
     SQL: SELECT p.name, p.price 
FROM orders o 
JOIN products p ON o.product = 
     Complexity: moderate (3.5)
     Gold result: [('Laptop', 999.99)]
     Got result : []  err=None
  ❌  Find users who have placed more than 1 order
     SQL: SELECT u.id, u.name, COUNT(o.id) as order_count
FROM users u
JOIN orde
     Complexity: complex (6.5)
     Gold result: [(1, 3), (3, 2)]
     Got result : [(1, 'Alice', 3), (3, 'Carol', 2)]  err=None
  ✅  What is the average age of users in ea

## 8. Aggregated Text-to-SQL Scorecard

In [9]:
print('\n' + '='*60)
print('  📊 TEXT-TO-SQL EVALUATION SCORECARD')
print('='*60)
print(f'''
  Metric                              Score
  ────────────────────────────────── ──────────
  Exact Match (string)                {exact_hits/len(sql_pairs):.1%}
  Avg Component Match                 {np.mean(comp_scores):.1%}
  Execution Accuracy (SQLite)         See Section 2
  Average VES                         {np.mean(ves_scores):.4f}
  Schema Linking Table Recall         {np.mean(sl_scores):.1%}
  Groq E2E Execution Accuracy        {correct}/{len(nl_questions)} = {correct/len(nl_questions):.1%}
''')


  📊 TEXT-TO-SQL EVALUATION SCORECARD

  Metric                              Score
  ────────────────────────────────── ──────────
  Exact Match (string)                20.0%
  Avg Component Match                 84.0%
  Execution Accuracy (SQLite)         See Section 2
  Average VES                         0.6639
  Schema Linking Table Recall         100.0%
  Groq E2E Execution Accuracy        3/7 = 42.9%

